In [ ]:
# =============================================================================
# FINAL SUMMARY & RESULTS
# =============================================================================

print("\n" + "="*70)
print("🎯 TRAINING SUMMARY")
print("="*70)

import json

eval_dir = WORKING_DIR / "evaluation"

with open(str(eval_dir / "resnet50_results.json"), 'r') as f:
    resnet_results = json.load(f)

with open(str(eval_dir / "efficientnet_b0_results.json"), 'r') as f:
    efficientnet_results = json.load(f)

# PrintResults
print("\n" + "─"*70)
print("RESNET50 RESULTS")
print("─"*70)
print(f"Test Accuracy:    {resnet_results['test_accuracy']:.4f} ({resnet_results['test_accuracy']*100:.2f}%)")
print(f"Precision:        {resnet_results['overall_precision']:.4f}")
print(f"Recall:           {resnet_results['overall_recall']:.4f}")
print(f"F1-Score:         {resnet_results['overall_f1']:.4f}")

print("\nTop 5 Classes (by F1-Score):")
sorted_classes = sorted(resnet_results['per_class_metrics'].items(),
                       key=lambda x: x[1]['f1-score'], reverse=True)[:5]
for class_name, metrics in sorted_classes:
    print(f"  • {class_name}: {metrics['f1-score']:.4f} (support: {metrics['support']})")

print("\n" + "─"*70)
print("EFFICIENTNETB0 RESULTS")
print("─"*70)
print(f"Test Accuracy:    {efficientnet_results['test_accuracy']:.4f} ({efficientnet_results['test_accuracy']*100:.2f}%)")
print(f"Precision:        {efficientnet_results['overall_precision']:.4f}")
print(f"Recall:           {efficientnet_results['overall_recall']:.4f}")
print(f"F1-Score:         {efficientnet_results['overall_f1']:.4f}")

print("\nTop 5 Classes (by F1-Score):")
sorted_classes = sorted(efficientnet_results['per_class_metrics'].items(),
                       key=lambda x: x[1]['f1-score'], reverse=True)[:5]
for class_name, metrics in sorted_classes:
    print(f"  • {class_name}: {metrics['f1-score']:.4f} (support: {metrics['support']})")

# Output directory summary
print("\n" + "─"*70)
print("OUTPUT FILES")
print("─"*70)

models_dir = WORKING_DIR / "models"
print("\n📁 Models (in /kaggle/working/models/):")
for model_file in sorted(models_dir.glob("*")):
    size_mb = model_file.stat().st_size / (1024*1024)
    print(f"  • {model_file.name} ({size_mb:.2f} MB)")

eval_dir = WORKING_DIR / "evaluation"
print("\n📊 Evaluation Results (in /kaggle/working/evaluation/):")
for eval_file in sorted(eval_dir.glob("*")):
    print(f"  • {eval_file.name}")

print("\n" + "="*70)
print("✓ TRAINING PIPELINE COMPLETE!")
print("="*70)

print("""
📋 NEXT STEPS:
1. Check the visualizations in /kaggle/working/evaluation/
2. Use trained models from /kaggle/working/models/
3. Download results from Output tab
4. Share metrics and visualizations in your project report

🔗 Model Files:
  - resnet50_best.h5 (best model on validation)
  - resnet50_final.h5 (final trained model)
  - efficientnet_b0_best.h5 (best model on validation)
  - efficientnet_b0_final.h5 (final trained model)
  - class_names.pkl (class labels for inference)

📊 Evaluation Files:
  - resnet50_results.json (detailed metrics)
  - efficientnet_b0_results.json (detailed metrics)
  - resnet50_confusion_matrix.png (confusion matrix)
  - efficientnet_b0_confusion_matrix.png (confusion matrix)
  - resnet50_training_curves.png (accuracy & loss curves)
  - efficientnet_b0_training_curves.png (accuracy & loss curves)
  - model_comparison.png (side-by-side comparison)
""")


In [ ]:
# =============================================================================
# STEP 8: VISUALIZATIONS & TRAINING CURVES
# =============================================================================

print("\n" + "="*70)
print("📈 GENERATING VISUALIZATIONS")
print("="*70)

import pickle
import matplotlib.pyplot as plt
import seaborn as sns

eval_dir = WORKING_DIR / "evaluation"

# Load training histories
with open(WORKING_DIR / "models" / "resnet50_history.pkl", 'rb') as f:
    resnet_history = pickle.load(f)

with open(WORKING_DIR / "models" / "efficientnet_b0_history.pkl", 'rb') as f:
    efficientnet_history = pickle.load(f)

# Plot training curves for ResNet50
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(resnet_history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(resnet_history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('ResNet50 - Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(resnet_history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(resnet_history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('ResNet50 - Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(eval_dir / "resnet50_training_curves.png"), dpi=100, bbox_inches='tight')
plt.close()
print("  ✓ Saved ResNet50 training curves")

# Plot training curves for EfficientNetB0
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(efficientnet_history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(efficientnet_history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('EfficientNetB0 - Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(efficientnet_history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(efficientnet_history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('EfficientNetB0 - Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(eval_dir / "efficientnet_b0_training_curves.png"), dpi=100, bbox_inches='tight')
plt.close()
print("  ✓ Saved EfficientNetB0 training curves")

# Load evaluation results for comparison
import json

with open(str(eval_dir / "resnet50_results.json"), 'r') as f:
    resnet_results = json.load(f)

with open(str(eval_dir / "efficientnet_b0_results.json"), 'r') as f:
    efficientnet_results = json.load(f)

# Model comparison bar chart
models = ['ResNet50', 'EfficientNetB0']
metrics = ['test_accuracy', 'overall_precision', 'overall_recall', 'overall_f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(metric_labels))
width = 0.35

resnet_values = [resnet_results[m] for m in metrics]
efficientnet_values = [efficientnet_results[m] for m in metrics]

bars1 = ax.bar(x - width/2, resnet_values, width, label='ResNet50')
bars2 = ax.bar(x + width/2, efficientnet_values, width, label='EfficientNetB0')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.legend()
ax.set_ylim([0.5, 1.0])
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(str(eval_dir / "model_comparison.png"), dpi=100, bbox_inches='tight')
plt.close()
print("  ✓ Saved model comparison chart")

print("\n✓ Visualizations complete!")


In [ ]:
# =============================================================================
# STEP 7: MODEL EVALUATION
# =============================================================================

print("\n" + "="*70)
print("📊 MODEL EVALUATION")
print("="*70)

class ModelEvaluator:
    def __init__(self, models_dir, test_dir):
        self.models_dir = Path(models_dir)
        self.test_dir = Path(test_dir)
        self.eval_dir = WORKING_DIR / "evaluation"
        self.eval_dir.mkdir(exist_ok=True)
    
    def evaluate_on_test_set(self, model_name):
        import pickle
        model = tf.keras.models.load_model(str(self.models_dir / f"{model_name}_best.h5"))
        
        test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
        test_generator = test_datagen.flow_from_directory(
            self.test_dir,
            target_size=(224, 224),
            batch_size=32,
            class_mode='categorical',
            shuffle=False
        )
        
        loss, accuracy = model.evaluate(test_generator, verbose=0)
        
        with open(str(self.models_dir / "class_names.pkl"), 'rb') as f:
            class_names = pickle.load(f)
        
        y_true = test_generator.classes
        y_pred = model.predict(test_generator, verbose=0).argmax(axis=1)
        
        from sklearn.metrics import (
            accuracy_score, precision_score, recall_score, f1_score,
            confusion_matrix, classification_report
        )
        
        results = {
            'model_name': model_name,
            'test_loss': float(loss),
            'test_accuracy': float(accuracy),
            'overall_precision': float(precision_score(y_true, y_pred, average='weighted')),
            'overall_recall': float(recall_score(y_true, y_pred, average='weighted')),
            'overall_f1': float(f1_score(y_true, y_pred, average='weighted')),
            'per_class_metrics': {}
        }
        
        report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
        
        for i, class_name in enumerate(class_names):
            if str(i) in report:
                results['per_class_metrics'][class_name] = {
                    'precision': float(report[str(i)]['precision']),
                    'recall': float(report[str(i)]['recall']),
                    'f1-score': float(report[str(i)]['f1-score']),
                    'support': int(report[str(i)]['support'])
                }
        
        return results, y_true, y_pred, class_names
    
    def save_results(self, results, filename):
        import json
        with open(str(self.eval_dir / filename), 'w') as f:
            json.dump(results, f, indent=2)

# Load class names
import pickle
with open(WORKING_DIR / "models" / "class_names.pkl", 'rb') as f:
    class_names = pickle.load(f)

test_dir = WORKING_DIR / "dataset_split" / "test"

evaluator = ModelEvaluator(WORKING_DIR / "models", test_dir)

# Evaluate ResNet50
print("\nEvaluating ResNet50...")
resnet_results, resnet_y_true, resnet_y_pred, _ = evaluator.evaluate_on_test_set("resnet50")
evaluator.save_results(resnet_results, "resnet50_results.json")

print(f"  ✓ Accuracy: {resnet_results['test_accuracy']:.4f}")
print(f"  ✓ Precision: {resnet_results['overall_precision']:.4f}")
print(f"  ✓ Recall: {resnet_results['overall_recall']:.4f}")
print(f"  ✓ F1-Score: {resnet_results['overall_f1']:.4f}")

# Evaluate EfficientNetB0
print("\nEvaluating EfficientNetB0...")
efficientnet_results, efficientnet_y_true, efficientnet_y_pred, _ = evaluator.evaluate_on_test_set("efficientnet_b0")
evaluator.save_results(efficientnet_results, "efficientnet_b0_results.json")

print(f"  ✓ Accuracy: {efficientnet_results['test_accuracy']:.4f}")
print(f"  ✓ Precision: {efficientnet_results['overall_precision']:.4f}")
print(f"  ✓ Recall: {efficientnet_results['overall_recall']:.4f}")
print(f"  ✓ F1-Score: {efficientnet_results['overall_f1']:.4f}")

# Generate confusion matrices
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

for model_name, y_pred in [("ResNet50", resnet_y_pred), ("EfficientNetB0", efficientnet_y_pred)]:
    cm = confusion_matrix(resnet_y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{model_name} - Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(str(WORKING_DIR / "evaluation" / f"{model_name.lower().replace(' ', '_')}_confusion_matrix.png"), dpi=100, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved {model_name} confusion matrix")

print("\n✓ Evaluation complete!")


In [ ]:
# =============================================================================
# STEP 6: MODEL TRAINING (ResNet50 + EfficientNetB0)
# =============================================================================

print("\n" + "="*70)
print("🤖 MODEL TRAINING")
print("="*70)

class ModelTrainer:
    def __init__(self, img_size=224, batch_size=32, epochs=30):
        self.img_size = img_size
        self.batch_size = batch_size
        self.epochs = epochs
        self.models_dir = WORKING_DIR / "models"
        self.models_dir.mkdir(exist_ok=True)
    
    @staticmethod
    def get_class_names(data_dir):
        return sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
    
    def build_resnet50(self, num_classes):
        base_model = tf.keras.applications.ResNet50(
            input_shape=(self.img_size, self.img_size, 3),
            include_top=False,
            weights='imagenet'
        )
        base_model.trainable = False
        
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.img_size, self.img_size, 3)),
            tf.keras.layers.Rescaling(1./255),
            base_model,
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(num_classes, activation='softmax')
        ])
        return model
    
    def build_efficientnet_b0(self, num_classes):
        base_model = tf.keras.applications.EfficientNetB0(
            input_shape=(self.img_size, self.img_size, 3),
            include_top=False,
            weights='imagenet'
        )
        base_model.trainable = False
        
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.img_size, self.img_size, 3)),
            tf.keras.layers.Rescaling(1./255),
            base_model,
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(num_classes, activation='softmax')
        ])
        return model
    
    def train_model(self, model, train_generator, val_generator, model_name):
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-7,
                verbose=1
            ),
            tf.keras.callbacks.ModelCheckpoint(
                str(self.models_dir / f"{model_name}_best.h5"),
                monitor='val_accuracy',
                save_best_only=True,
                verbose=0
            )
        ]
        
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        print(f"\nTraining {model_name}...")
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=self.epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        model.save(str(self.models_dir / f"{model_name}_final.h5"))
        
        import pickle
        with open(str(self.models_dir / f"{model_name}_history.pkl"), 'wb') as f:
            pickle.dump(history.history, f)
        
        return history

# Prepare data generators
train_path = WORKING_DIR / "dataset_split" / "train"
val_path = WORKING_DIR / "dataset_split" / "val"

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

class_names = sorted([d.name for d in train_path.iterdir() if d.is_dir()])
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")

# Initialize trainer
trainer = ModelTrainer(img_size=224, batch_size=32, epochs=30)

# Save class names
import pickle
with open(trainer.models_dir / "class_names.pkl", 'wb') as f:
    pickle.dump(class_names, f)

# Build and train ResNet50
print("\n" + "-"*70)
print("Training ResNet50...")
resnet_model = trainer.build_resnet50(num_classes)
print(f"ResNet50 parameters: {resnet_model.count_params():,}")
resnet_history = trainer.train_model(resnet_model, train_generator, val_generator, "resnet50")

# Reset generators
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)
val_generator = val_datagen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Build and train EfficientNetB0
print("\n" + "-"*70)
print("Training EfficientNetB0...")
efficientnet_model = trainer.build_efficientnet_b0(num_classes)
print(f"EfficientNetB0 parameters: {efficientnet_model.count_params():,}")
efficientnet_history = trainer.train_model(efficientnet_model, train_generator, val_generator, "efficientnet_b0")

print("\n✓ Model training complete!")


In [ ]:
# =============================================================================
# STEP 5: DATA AUGMENTATION (Training Set Only)
# =============================================================================

print("\n" + "="*70)
print("🔄 DATA AUGMENTATION (Training Set Only)")
print("="*70)

class DataAugmentation:
    """Apply various data augmentation techniques"""
    
    @staticmethod
    def rotate(image, angle_range=(-15, 15)):
        angle = random.uniform(angle_range[0], angle_range[1])
        return image.rotate(angle, expand=True, fillcolor='white')
    
    @staticmethod
    def flip(image, horizontal=True):
        if random.random() > 0.5 and horizontal:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
        return image
    
    @staticmethod
    def brightness_variation(image, factor_range=(0.7, 1.3)):
        factor = random.uniform(factor_range[0], factor_range[1])
        enhancer = ImageEnhance.Brightness(image)
        return enhancer.enhance(factor)
    
    @staticmethod
    def zoom(image, zoom_range=(0.8, 1.0)):
        zoom_factor = random.uniform(zoom_range[0], zoom_range[1])
        width, height = image.size
        new_width = int(width * zoom_factor)
        new_height = int(height * zoom_factor)
        
        left = (width - new_width) // 2
        top = (height - new_height) // 2
        right = left + new_width
        bottom = top + new_height
        
        cropped = image.crop((left, top, right, bottom))
        return cropped.resize((width, height), Image.Resampling.LANCZOS)
    
    @staticmethod
    def random_crop(image, crop_percent=0.1):
        width, height = image.size
        crop_width = int(width * (1 - crop_percent))
        crop_height = int(height * (1 - crop_percent))
        
        left = random.randint(0, width - crop_width)
        top = random.randint(0, height - crop_height)
        
        cropped = image.crop((left, top, left + crop_width, top + crop_height))
        return cropped.resize((width, height), Image.Resampling.LANCZOS)
    
    @staticmethod
    def augment_image(image_path, augmentation_count=2):
        try:
            image = Image.open(image_path).convert('RGB')
            augmented_images = [image]
            
            for _ in range(augmentation_count):
                aug_image = image.copy()
                
                if random.random() > 0.5:
                    aug_image = DataAugmentation.rotate(aug_image)
                if random.random() > 0.5:
                    aug_image = DataAugmentation.flip(aug_image)
                if random.random() > 0.5:
                    aug_image = DataAugmentation.brightness_variation(aug_image)
                if random.random() > 0.5:
                    aug_image = DataAugmentation.zoom(aug_image)
                if random.random() > 0.5:
                    aug_image = DataAugmentation.random_crop(aug_image)
                
                augmented_images.append(aug_image)
            
            return augmented_images
        except:
            return None

# Apply augmentation
from PIL import ImageEnhance

print("Augmenting training data...")
train_path = Path(WORKING_DIR) / "dataset_split" / "train"
augmented_count = 0

if train_path.exists():
    for class_folder in sorted(train_path.iterdir()):
        if not class_folder.is_dir():
            continue
        
        images = [f for f in class_folder.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.jfif', '.png']]
        original_count = len(images)
        
        for idx, img_path in enumerate(images, 1):
            augmented = DataAugmentation.augment_image(img_path, augmentation_count=2)
            
            if augmented:
                stem = img_path.stem
                for aug_idx, aug_image in enumerate(augmented[1:], 1):
                    new_filename = f"{stem}_aug_{aug_idx}{img_path.suffix}"
                    new_path = class_folder / new_filename
                    aug_image.save(new_path, quality=95)
                    augmented_count += 1
        
        new_count = len([f for f in class_folder.iterdir() if f.is_file()])
        print(f"  ✓ {class_folder.name}: {original_count} → {new_count} images (+{new_count - original_count})")

print(f"\n✓ Augmentation complete! Created {augmented_count} augmented images.")


In [ ]:
# =============================================================================
# STEP 4: DATASET SPLITTING (75% Train / 15% Val / 10% Test)
# =============================================================================

print("\n" + "="*70)
print("🔀 SPLITTING DATASET (75% Train / 15% Val / 10% Test)")
print("="*70)

def create_split_structure(source_dir, output_dir, train_ratio=0.75, val_ratio=0.15, test_ratio=0.10):
    """Split dataset into train/val/test directories"""
    
    source_path = Path(source_dir)
    output_path = Path(output_dir)
    
    # Create output structure
    train_dir = output_path / "train"
    val_dir = output_path / "val"
    test_dir = output_path / "test"
    
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all class folders
    classes = [d for d in source_path.iterdir() 
               if d.is_dir() and not d.name.startswith('.')]
    
    print(f"\nFound {len(classes)} classes\n")
    
    stats = defaultdict(lambda: {"train": 0, "val": 0, "test": 0, "total": 0})
    
    for class_folder in sorted(classes):
        class_name = class_folder.name
        
        # Get all images
        images = [f for f in class_folder.iterdir() 
                 if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.jfif', '.png', '.bmp']]
        
        if not images:
            continue
        
        # Shuffle and split
        random.shuffle(images)
        total = len(images)
        
        train_count = int(total * train_ratio)
        val_count = int(total * val_ratio)
        
        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]
        
        # Create class folders in each split
        train_class_dir = train_dir / class_name
        val_class_dir = val_dir / class_name
        test_class_dir = test_dir / class_name
        
        train_class_dir.mkdir(exist_ok=True)
        val_class_dir.mkdir(exist_ok=True)
        test_class_dir.mkdir(exist_ok=True)
        
        # Copy files
        for img in train_images:
            shutil.copy2(img, train_class_dir / img.name)
        for img in val_images:
            shutil.copy2(img, val_class_dir / img.name)
        for img in test_images:
            shutil.copy2(img, test_class_dir / img.name)
        
        # Statistics
        stats[class_name]["train"] = len(train_images)
        stats[class_name]["val"] = len(val_images)
        stats[class_name]["test"] = len(test_images)
        stats[class_name]["total"] = total
        
        print(f"{class_name:30} Train: {len(train_images):3} | Val: {len(val_images):2} | Test: {len(test_images):2} | Total: {total:3}")
    
    # Summary
    print("\n" + "="*70)
    total_train = sum(s["train"] for s in stats.values())
    total_val = sum(s["val"] for s in stats.values())
    total_test = sum(s["test"] for s in stats.values())
    total_all = total_train + total_val + total_test
    
    print(f"TOTAL: Train: {total_train} | Val: {total_val} | Test: {total_test} | All: {total_all}")
    print(f"Percentages: Train: {total_train/total_all*100:.1f}% | Val: {total_val/total_all*100:.1f}% | Test: {total_test/total_all*100:.1f}%")
    print("="*70)
    
    return stats

# Run split
source_dir = Path(WORKING_DIR) / "dataset_raw"
output_dir = Path(WORKING_DIR) / "dataset_split"

if source_dir.exists():
    split_stats = create_split_structure(str(source_dir), str(output_dir))
    print("\n✓ Dataset split complete!")
else:
    print(f"❌ Source directory not found: {source_dir}")


In [ ]:
# =============================================================================
# STEP 3: COPY DATASET - Copy from Kaggle input to working directory
# =============================================================================

print("\n" + "="*70)
print("📂 COPYING DATASET FROM KAGGLE INPUT")
print("="*70)

# Copy dataset to working directory
if os.path.exists(DATASET_DIR):
    import shutil
    dest_dir = os.path.join(WORKING_DIR, "dataset_raw")
    
    if not os.path.exists(dest_dir):
        print(f"\nCopying dataset from {DATASET_DIR}...")
        shutil.copytree(DATASET_DIR, dest_dir)
        print(f"✓ Dataset copied to {dest_dir}")
    else:
        print(f"✓ Dataset already exists at {dest_dir}")
    
    # List class folders
    class_folders = [d for d in os.listdir(dest_dir) if os.path.isdir(os.path.join(dest_dir, d))]
    print(f"\n✓ Found {len(class_folders)} class folders:")
    
    # Count images per class
    image_counts = {}
    for class_name in sorted(class_folders):
        class_path = os.path.join(dest_dir, class_name)
        images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.jfif', '.png', '.bmp'))]
        image_counts[class_name] = len(images)
        print(f"  - {class_name}: {len(images)} images")
    
    total_images = sum(image_counts.values())
    print(f"\n✓ Total images: {total_images}")
    
    # Verify we have the data
    if total_images == 0:
        print("⚠️  No images found! Check dataset folder structure.")
    else:
        print("✓ Dataset ready for processing!")
else:
    print(f"❌ Dataset not found at {DATASET_DIR}")


In [ ]:
# =============================================================================
# STEP 2: CONFIGURATION & PATHS
# =============================================================================

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paths
KAGGLE_INPUT_DIR = "/kaggle/input"
KAGGLE_WORKING_DIR = "/kaggle/working"

# Find your dataset slug (list all input directories)
input_dirs = os.listdir(KAGGLE_INPUT_DIR)
print("\n📁 Available datasets in /kaggle/input:")
for d in input_dirs:
    if os.path.isdir(os.path.join(KAGGLE_INPUT_DIR, d)):
        print(f"  - {d}")

# Set your dataset slug (CHANGE THIS to match your dataset folder name)
# Example: "yourusername/pakistani-politicians-face-dataset"
# Look for folder that starts with your username or contains "politician"
DATASET_SLUG = None
for d in input_dirs:
    if 'politician' in d.lower() or 'face' in d.lower() or 'pakistan' in d.lower():
        DATASET_SLUG = d
        break

if DATASET_SLUG is None:
    DATASET_SLUG = input_dirs[0] if input_dirs else None
    
print(f"\n✓ Using dataset: {DATASET_SLUG}")
DATASET_DIR = os.path.join(KAGGLE_INPUT_DIR, DATASET_SLUG)
WORKING_DIR = KAGGLE_WORKING_DIR

print(f"  Input directory: {DATASET_DIR}")
print(f"  Working directory: {WORKING_DIR}")

# Change to working directory
os.chdir(WORKING_DIR)
print(f"\n✓ Changed to: {WORKING_DIR}")


In [ ]:
# =============================================================================
# STEP 1: INSTALL & IMPORT DEPENDENCIES
# =============================================================================

# Install required packages (if needed)
import subprocess
import sys

packages = ['tensorflow', 'scikit-learn', 'pillow', 'matplotlib', 'seaborn']
for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Core imports
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path
import json
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import random
import shutil
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("\n✓ All imports successful!")
print(f"✓ TensorFlow version: {tf.__version__}")
print(f"✓ GPU Available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")


# Pakistani Politician Image Classification - Kaggle Training Pipeline
## Category B Project 2: CNN-Based Face Recognition

**Objective:** Train ResNet50 and EfficientNetB0 models on 1,551 facial images of 16 Pakistani political figures.

**Dataset:** Kaggle - pakistani-politicians-face-dataset (or your dataset slug)

**Models:** ResNet50 (25M params) + EfficientNetB0 (5.3M params)

**Target:** ≥90% accuracy with confusion matrix, per-class metrics, and training curves

**Notebook Workflow:**
1. ✓ Install dependencies
2. ✓ Copy dataset from Kaggle input
3. ✓ Split into train/val/test (75/15/10)
4. ✓ Apply data augmentation
5. ✓ Train both models
6. ✓ Evaluate and compute metrics
7. ✓ Generate visualizations
8. ✓ Download results

**⚠️ Important:** Enable GPU in Notebook Settings before running!
